In [2]:
import json
from heapq import heappush, heappop
from typing import Literal

In [3]:
JSON_MAPS_FILE = 'maps.json'
MAP = 'mapa-4'

In [5]:
def find_positions(chosen_map: list[list[str]]) -> dict[Literal['I', 'F'], tuple[int,int]]:
    res = {}
    for ir, row in enumerate(chosen_map):
        for ic, col in enumerate(row):
            if col == 'I':
                assert res.get('I') == None
                res['I'] = (ir, ic)
            elif col == 'F':
                assert res.get('F') == None
                res['F'] = (ir, ic)
    return res

In [6]:
def is_blocked(chosen_map: list[list[str]], pos: tuple[int, int]) -> bool:
    return chosen_map[pos[0]][pos[1]] == '*'

In [7]:
def get_move_cost(chosen_map: list[list[str]], pos: tuple[int, int]) -> int:
    cell = chosen_map[pos[0]][pos[1]]
    return 0 if cell == 'F' else int(cell)

In [8]:
def euclidian(p1: tuple[int,int], p2: tuple[int,int]) -> float:
    return ((p1[0] - p2[0])**2 + (p1[1] - p2[1])**2)**0.5

def manhattan(p1: tuple[int,int], p2: tuple[int,int]) -> float:
    return abs(p1[0] - p2[0]) + abs(p1[1] - p2[1])

In [9]:
def create_node(position: tuple[int, int], g: float = float('inf'), 
                h: float = 0.0, parent: dict = None) -> dict:
    return {
        'position': position,
        'g': g,
        'h': h,
        'f': g + h,
        'parent': parent
    }

In [10]:
def above(pos: tuple[int,int], max_row: int, max_col: int) -> tuple[int,int] | None:
    if pos[0] <= 0:
        return None
    return (pos[0] - 1, pos[1])

def below(pos: tuple[int,int], max_row: int, max_col: int) -> tuple[int,int] | None:
    if pos[0] >= max_row - 1:
        return None
    return (pos[0] + 1, pos[1])

def left(pos: tuple[int,int], max_row: int, max_col: int) -> tuple[int,int] | None:
    if pos[1] <= 0:
        return None
    return (pos[0], pos[1] - 1)

def right(pos: tuple[int,int], max_row: int, max_col: int) -> tuple[int,int] | None:
    if pos[1] >= max_col - 1:
        return None
    return (pos[0], pos[1] + 1)

In [11]:
def make_path(open_dict: dict, start_pos: tuple[int, int], goal_pos: tuple[int, int]) -> tuple[list[dict], int]:
    path = []
    curr_pos = goal_pos
    total_cost = open_dict[goal_pos]['g']
    while curr_pos != start_pos:
        path.append(open_dict[curr_pos])
        curr_pos = open_dict[curr_pos]['parent']['position']
    path.append(open_dict[start_pos])
    return path[::-1], total_cost # reverse the path and return total cost

In [12]:
with open(JSON_MAPS_FILE, 'r') as f:
    maps = json.load(f)

In [13]:
chosen_map = maps[MAP]
num_rows_map = len(chosen_map)
num_cols_map = len(chosen_map[0])

In [14]:
positions = find_positions(chosen_map)
print(positions)

start = positions['I']
goal = positions['F']

{'I': (0, 0), 'F': (2, 3)}


In [15]:
start_node = create_node(position=start,
                         g=0,
                         h=manhattan(start, goal))

# Initialize open and closed sets
open_ = [(start_node['f'], start)]
open_dict = {start: start_node}
closed_= set()

while len(open_) > 0:
    _, current_pos = heappop(open_)
    current_node = open_dict[current_pos]

    if current_pos == goal:
        break
    
    # If the current position is already in the closed set, skip it
    # Obs.: this can happen because we might have added the same position
    # multiple times to the open set with different g values
    if current_pos in closed_:
        continue

    closed_.add(current_pos)

    neighbors = filter(lambda x: x is not None, [
        above(current_pos, num_rows_map, num_cols_map),
        below(current_pos, num_rows_map, num_cols_map),
        left(current_pos, num_rows_map, num_cols_map),
        right(current_pos, num_rows_map, num_cols_map)
    ])
    for n_pos in neighbors:
        if is_blocked(chosen_map, n_pos) or n_pos in closed_:
            continue

        n_node = open_dict.get(n_pos)
        move_cost = get_move_cost(chosen_map, n_pos)
        temp_g = current_node['g'] + move_cost

        if n_node is None or temp_g < n_node['g']:
            new_node = create_node(position=n_pos,
                                   g=temp_g,
                                   h=manhattan(n_pos, goal),
                                   parent=current_node)
            open_dict[n_pos] = new_node
            heappush(open_, (new_node['f'], n_pos))

In [16]:
path, total_cost = make_path(open_dict, start, goal)
print('Path:')
for node in path:
    print(' ', node['position'], 'g:', node['g'], 'h:', node['h'], 'f:', node['f'])
print('Total cost:', total_cost)

Path:
  (0, 0) g: 0 h: 5 f: 5
  (1, 0) g: 1 h: 4 f: 5
  (2, 0) g: 2 h: 3 f: 5
  (2, 1) g: 3 h: 2 f: 5
  (2, 2) g: 4 h: 1 f: 5
  (2, 3) g: 4 h: 0 f: 4
Total cost: 4
